In [2]:
import os
import re
from PIL import Image
import matplotlib.pyplot as plt

ROOT = "./"

pat = re.compile(r"(pinn_(loss|pred)_round_(\d+)_seed_42)\.png")

for sub in os.listdir(ROOT):
    subdir = os.path.join(ROOT, sub)
    if not os.path.isdir(subdir):
        continue

    loss_imgs = {}
    pred_imgs = {}

    for f in os.listdir(subdir):
        m = pat.match(f)
        if m:
            name, kind, round_id = m.group(1), m.group(2), int(m.group(3))
            path = os.path.join(subdir, f)
            if kind == "loss":
                loss_imgs[round_id] = path
            else:
                pred_imgs[round_id] = path

    if not loss_imgs or not pred_imgs:
        print(f"Skip {sub} (missing images)")
        continue

    rounds = sorted(loss_imgs.keys())

    # -------------------------
    # 创建图（2×4），无边距版
    # -------------------------
    fig, axes = plt.subplots(
        2, 4,
        figsize=(12, 4),
        gridspec_kw=dict(wspace=0.01, hspace=0.01)   # 减到极小
    )

    for ax_row in axes:
        for ax in ax_row:
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_frame_on(False)

    # 第一行：loss
    for i, r in enumerate(rounds):
        img = Image.open(loss_imgs[r])
        axes[0][i].imshow(img)
        axes[0][i].set_title(f"Loss {r}", fontsize=12, pad=1)

    # 第二行：pred
    for i, r in enumerate(rounds):
        img = Image.open(pred_imgs[r])
        axes[1][i].imshow(img)
        axes[1][i].set_title(f"Pred {r}", fontsize=12, pad=1)

    # 去掉四周 margin
    plt.subplots_adjust(
        left=0, right=1, top=1, bottom=0,
        wspace=0.01, hspace=0.01
    )

    out_path = os.path.join(subdir, "merged.png")
    plt.savefig(out_path, dpi=200, bbox_inches="tight", pad_inches=0)
    plt.close()
    print(f"Saved merged: {out_path}")


Saved merged: ./burgers\merged.png
Saved merged: ./pendulum\merged.png
Saved merged: ./possion-1d\merged.png
Saved merged: ./van_der_pol\merged.png
